In [ ]:
# %pip install -qqq pandas numpy matplotlib seaborn ucimlrepo

In [ ]:
from datetime import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo 

## 1. データ読み込み

下記の公開データを利用  
https://archive.ics.uci.edu/dataset/352/online+retail

In [ ]:
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 

In [ ]:
df_raw = online_retail.data.original

In [ ]:
df_raw.head()

## 2. データの概要確認

In [ ]:
# サンプル数を確認
df_raw.shape

In [ ]:
# データ型の確認
df_raw.dtypes

In [ ]:
# 欠損値確認
df_raw.isnull().sum()

# →DescriptionとCustomerIDに欠損値あり
# rfm分析とmba分析の時には弾く
# Description: 1454
# CustomerID: 135080

In [ ]:
# データ概要の確認
# InvoiceNo	ID	Categorical	a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation
df_raw["InvoiceNo"].describe()

In [ ]:
# StockCode	ID	Categorical	a 5-digit integral number uniquely assigned to each distinct product
df_raw["StockCode"].describe()

In [ ]:
# Description	Feature	Categorical	product name
df_raw["Description"].describe()

# →欠損値あり

In [ ]:
# Quantity	Feature	Integer	the quantities of each product (item) per transaction
df_raw["Quantity"].describe()

In [ ]:
# InvoiceDate	Feature	Date	the day and time when each transaction was generated

df_raw["InvoiceDate"].describe()

In [ ]:
# UnitPrice	Feature	Continuous	product price per unit	sterling
df_raw["UnitPrice"].describe()


In [ ]:
# CustomerID	Feature	Categorical	a 5-digit integral number uniquely assigned to each customer
df_raw["CustomerID"].describe()

# →欠損値あり

In [ ]:
# Country	Feature	Categorical	the name of the country where each customer resides
df_raw["Country"].describe()

## 3. データの中身の確認

In [ ]:
df_data = df_raw.copy()

### 3.1 InvoiceNoの処理

In [ ]:
df_data["InvoiceNo"].value_counts()

In [ ]:
# キャンセルデータはInvoiceNoの先頭にCが付いている。
# 今回はキャンセルフラグを作る

df_data["cancelled_flag"] = df_data["InvoiceNo"].astype(str).str.upper().str.startswith("C")

In [ ]:
df_data.head(5)

### 3.2 Quantity

In [ ]:
# キャンセルされてないデータだけを確認する
df_data["Quantity"][df_data["cancelled_flag"] == False].describe()

In [ ]:
# グラフかしてみる

data_filtered = df_data[df_data["cancelled_flag"] == False]
sns.histplot(
    data=data_filtered,
    x="Quantity",
    binwidth=1
)

# データの大部分（9割以上）が収まる範囲に画面をズームする
plt.xlim(-5, 50) 
plt.ylabel("Frequency")
plt.title("Histogram of Quantity")

plt.show()


In [ ]:
# Quantityが0未満のレコードが何個あるか確認
len(df_data[(df_data["cancelled_flag"] == False) & (df_data["Quantity"]<0)])

### 3.3 UnitPrice

In [ ]:
# キャンセルされてないデータだけを確認する
df_data["UnitPrice"][df_data["cancelled_flag"] == False].describe()

In [ ]:
data_filtered = df_data[df_data["cancelled_flag"] == False]
sns.histplot(
    data=data_filtered,
    x="UnitPrice",
    binwidth=0.2
)

plt.xlim(-1, 15) 

plt.ylabel("Frequency")
plt.title("Histogram of UnitPrice")
plt.show()

In [ ]:
# UnitPriceが0以下のレコードが何個あるか確認
len(df_data[(df_data["cancelled_flag"] == False) & (df_data["UnitPrice"]<=0)])

### 3.5 InvoiceDate

In [ ]:
df_data["InvoiceDate"].head(3)

In [ ]:
# strをdatetimeに変換
date_format = "%m/%d/%Y %H:%M"

df_data["dt_invoice_date"] = pd.to_datetime(
    df_data["InvoiceDate"],
    format=date_format
)

In [ ]:
# 期間の始まりを確認
df_data["dt_invoice_date"].min()

In [ ]:
# 期間の終わりを確認
df_data["dt_invoice_date"].max()

### 3.6 Country

In [ ]:
df_data["Country"].value_counts()